# 01 — Explore endpoints

**Notebook yang kamu buka pertama kali setiap kali ada yang rusak.**

Tujuannya satu: mengirim **satu** request dan melihat struktur response mentahnya.
Tidak ada loop, tidak ada penulisan massal ke database, tidak ada scraping volume.

### Prasyarat

Sebelum menjalankan notebook ini kamu harus sudah:

1. Mengikuti `docs/CAPTURE_HEADERS.md` langkah A-D (capture di Chrome DevTools).
2. Menjalankan:
   ```powershell
   python scripts/curl_to_config.py capture_page1.txt capture_page2.txt --keyword "air fryer"
   ```
3. Memastikan `config/gql_capture.yaml` dan `.env` terbentuk.

Kalau belum, sel di bawah akan gagal dengan pesan yang menyebutkan langkah mana yang kurang —
itu memang perilaku yang diinginkan, bukan bug.

### Keamanan

Notebook ini **tidak pernah mencetak nilai cookie**. Yang ditampilkan hanya ada/tidaknya
dan panjangnya. Kalau kamu mau share notebook ini, tetap `Kernel > Restart & Clear Output` dulu:
output response mentah bisa mengandung data sesi.

In [1]:
import json
import os
import sys
from pathlib import Path

# Jalankan dari mana saja: cari root project (yang punya config.yaml).
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tokopedia_scraper.config import Config
from tokopedia_scraper.logging_setup import setup_logging

setup_logging("INFO", ROOT / "logs" / "explore.log", force=True)

cfg = Config.load(ROOT / "config.yaml")
print(f"root      : {ROOT}")
print(f"fetcher   : {cfg.fetcher}")
print(f"keywords  : {len(cfg.keywords)} -> {cfg.keywords[:3]} ...")
print(f"capture   : {cfg.graphql.capture_file}")

root      : c:\Users\Henry\Documents\KULIAH\lomba\AIC
fetcher   : graphql
keywords  : 20 -> ['kaos polos pria', 'hoodie oversize', 'sepatu sneakers pria'] ...
capture   : C:\Users\Henry\Documents\KULIAH\lomba\AIC\config\gql_capture.yaml


## 1. Apakah hasil capture masih ada dan masuk akal?

Sel ini tidak menyentuh jaringan. Ia hanya membaca `config/gql_capture.yaml` dan `.env`,
lalu melaporkan apa yang ada. Kalau ada `TODO` yang tersisa dari script capture,
sel ini yang menampilkannya.

In [2]:
import yaml

capture_path = cfg.graphql.capture_file
if not capture_path.exists():
    raise SystemExit(
        f"{capture_path} tidak ada.\n"
        f"Ikuti docs/CAPTURE_HEADERS.md lalu jalankan scripts/curl_to_config.py."
    )

capture = yaml.safe_load(capture_path.read_text(encoding="utf-8")) or {}
print(f"captured_at: {capture.get('captured_at')}\n")

for stage_name in ("search", "pdp"):
    stage = capture.get(stage_name)
    if not stage:
        print(f"[{stage_name}] BELUM DI-CAPTURE")
        if stage_name == "pdp":
            print("          -> stage 2 (deskripsi) belum bisa jalan. docs/CAPTURE_HEADERS.md langkah D.")
        print()
        continue

    print(f"[{stage_name}]")
    print(f"  endpoint  : {stage['endpoint']}")
    print(f"  operation : {stage['operation_name']}")
    print(f"  headers   : {list(stage.get('headers', {}))}")
    print(f"  paging    : {stage.get('paging')}")

    # Nilai rahasia tidak dicetak — hanya ada/tidak dan panjangnya.
    for header_name, env_name in (stage.get("secret_env") or {}).items():
        value = os.getenv(env_name, "")
        state = f"OK ({len(value)} chars)" if value.strip() else "KOSONG <-- isi di .env"
        print(f"  env       : {env_name:<28} {state}")

    for note in stage.get("notes", []):
        marker = "  !!" if str(note).startswith("TODO") else "   -"
        print(f"{marker} {note}")
    print()

captured_at: 2026-08-04T06:38:34+00:00

[search]
  endpoint  : https://gql.tokopedia.com/graphql/SearchProductV5Query
  operation : SearchProductV5Query
  headers   : ['accept', 'accept-language', 'bd-web-id', 'content-type', 'origin', 'priority', 'referer', 'sec-ch-ua', 'sec-ch-ua-mobile', 'sec-ch-ua-platform', 'sec-fetch-dest', 'sec-fetch-mode', 'sec-fetch-site', 'x-dark-mode', 'x-device', 'x-price-center', 'x-source', 'x-tkpd-lite-service', 'x-version']
  paging    : {'mode': 'start', 'rows_per_page': 60, 'param': 'next_offset_organic', 'anchor_start': 60, 'anchor_page': 2, 'also_templated': ['next_offset_organic_ad', 'page', 'start']}
  env       : TOKOPEDIA_COOKIE             OK (3538 chars)
  env       : TOKOPEDIA_UA                 OK (117 chars)
   - note: the two captures are 2 pages apart (2 -> 4), so the page size is derived by dividing the offset step by 2.
   - paging derived from capture diff: next_offset_organic 60 -> 180 (start)
   - paging derived from capture diff: ne

## 2. Satu request search

Ini satu-satunya sel di notebook ini yang menyentuh jaringan untuk stage 1.
Rate limiter tetap aktif, jadi ada jeda beberapa detik sebelum request keluar.

In [3]:
from tokopedia_scraper.fetchers.graphql import CaptureIncomplete, GraphQLFetcher
from tokopedia_scraper.ratelimit import FetchError

KEYWORD = cfg.keywords[0]
PAGE = 1

fetcher = GraphQLFetcher(cfg)
try:
    result = fetcher.search(KEYWORD, PAGE)
    print(f"OK  status={result.status}  fetcher={result.fetcher}")
    print(f"    meta={result.meta}")
except CaptureIncomplete as exc:
    print(f"CAPTURE BELUM LENGKAP:\n{exc}")
    raise
except FetchError as exc:
    print(f"REQUEST GAGAL (status={exc.status}):\n{exc}")
    print("\nLihat tabel 'Kapan harus capture ulang' di docs/CAPTURE_HEADERS.md.")
    raise

[13:45:10] WARNING  page 1 is below the captured anchor page 2 — the offset is extrapolated and may not line up    
                    with what the site expects.

OK  status=200  fetcher=graphql
    meta={'keyword': 'kaos polos pria', 'page': 1, 'rows': 60}


## 3. Bentuk response mentah

Response GraphQL Tokopedia dalam dan bercabang. Dua helper di bawah ini murni alat
eksplorasi — sengaja tinggal di notebook, bukan di `src/`, karena tidak dipakai pipeline.

`shape()` mencetak kerangka tipe. `find_record_arrays()` mencari array berisi dict —
kandidat daftar produk. **Path yang dilaporkan sel berikutnya adalah yang dibutuhkan
untuk menulis parser di fase E.**

In [4]:
def shape(obj, depth=0, max_depth=4, prefix=""):
    """Cetak kerangka tipe dari struktur JSON."""
    pad = "  " * depth
    if depth > max_depth:
        print(f"{pad}{prefix}...")
        return
    if isinstance(obj, dict):
        print(f"{pad}{prefix}dict({len(obj)})")
        for key, value in list(obj.items())[:15]:
            shape(value, depth + 1, max_depth, f"{key}: ")
    elif isinstance(obj, list):
        print(f"{pad}{prefix}list[{len(obj)}]")
        if obj:
            shape(obj[0], depth + 1, max_depth, "[0] ")
    else:
        preview = repr(obj)
        if len(preview) > 60:
            preview = preview[:60] + "..."
        print(f"{pad}{prefix}{type(obj).__name__} = {preview}")


def find_record_arrays(obj, path="", min_keys=3, found=None):
    """Cari array berisi dict — kandidat daftar produk."""
    found = [] if found is None else found
    if isinstance(obj, list):
        if obj and isinstance(obj[0], dict) and len(obj[0]) >= min_keys:
            found.append((path, len(obj), sorted(obj[0])[:20]))
        for i, item in enumerate(obj[:2]):
            find_record_arrays(item, f"{path}.{i}", min_keys, found)
    elif isinstance(obj, dict):
        for key, value in obj.items():
            find_record_arrays(value, f"{path}.{key}" if path else key, min_keys, found)
    return found


shape(result.payload)

list[1]
  [0] dict(1)
    data: dict(1)
      searchProductV5: dict(3)
        header: dict(10)
          totalData: ...
          responseCode: ...
          keywordProcess: ...
          keywordIntention: ...
          componentID: ...
          isQuerySafe: ...
          additionalParams: ...
          backendFilters: ...
          meta: ...
          __typename: ...
        data: dict(9)
          totalDataText: ...
          banner: ...
          redirection: ...
          related: ...
          suggestion: ...
          ticker: ...
          violation: ...
          products: ...
          __typename: ...
        __typename: str = 'SearchProductV5Response'


In [5]:
candidates = find_record_arrays(result.payload)
candidates.sort(key=lambda c: c[1], reverse=True)

print("Kandidat array produk (terbesar dulu):\n")
for path, count, keys in candidates[:10]:
    print(f"  {path}")
    print(f"    {count} item, field: {keys}\n")

print("^ Kirim output blok ini ke Claude. Ini yang dibutuhkan untuk menulis parsers.py.")

Kandidat array produk (terbesar dulu):

  .0.data.searchProductV5.data.products
    60 item, field: ['__typename', 'ads', 'applink', 'badge', 'category', 'freeShipping', 'id', 'labelGroups', 'labelGroupsVariant', 'mediaURL', 'meta', 'name', 'oldID', 'price', 'rating', 'shop', 'stock', 'ttsProductID', 'url', 'wishlist']

  .0.data.searchProductV5.data.products.0.labelGroups
    8 item, field: ['__typename', 'position', 'styles', 'title', 'type', 'url']

  .0.data.searchProductV5.data.products.1.labelGroups
    8 item, field: ['__typename', 'position', 'styles', 'title', 'type', 'url']

^ Kirim output blok ini ke Claude. Ini yang dibutuhkan untuk menulis parsers.py.


In [6]:
# Satu produk contoh, lengkap. Periksa: mana judul, harga, URL, product_id, shop?
if candidates:
    path = candidates[0][0]
    node = result.payload
    for part in path.split("."):
        node = node[int(part)] if part.isdigit() else node[part]
    print(f"contoh dari {path}[0]:\n")
    print(json.dumps(node[0], indent=2, ensure_ascii=False)[:4000])
else:
    print("Tidak ada array record ditemukan. Kemungkinan kamu meng-capture request")
    print("GraphQL yang salah — ulangi docs/CAPTURE_HEADERS.md langkah B4.")

TypeError: list indices must be integers or slices, not str

## 4. Simpan response mentah

Selalu simpan sebelum diparse. Kalau parser salah atau schema Tokopedia berubah,
file ini bisa diparse ulang tanpa scraping ulang — dan dipakai jadi fixture test.

In [ ]:
cfg.ensure_dirs()

sample_path = cfg.storage.export_dir / "raw_sample_search.json"
sample_path.write_text(
    json.dumps(result.payload, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"ditulis: {sample_path}  ({sample_path.stat().st_size:,} bytes)")

# Simpan juga ke tabel raw_responses, supaya alur yang sama teruji dari awal.
from tokopedia_scraper.storage import Storage

with Storage(cfg.storage.db_path) as store:
    store.save_raw("search", f"{KEYWORD}|{PAGE}", result.payload, keyword=KEYWORD, page=PAGE)
    print(f"raw_responses: {store.stats()['raw_responses']} baris")

## 5. Satu request PDP (stage 2)

Deskripsi produk tidak ada di hasil search. Sel ini menguji endpoint PDP.

Butuh stage `pdp` di `config/gql_capture.yaml` — `docs/CAPTURE_HEADERS.md` langkah D.
Isi `PRODUCT_URL` dengan URL produk asli (boleh ambil dari hasil search di atas).

In [ ]:
PRODUCT_URL = ""  # <-- isi, contoh: https://www.tokopedia.com/namatoko/nama-produk

if not PRODUCT_URL:
    print("PRODUCT_URL kosong — lewati sel ini sampai kamu isi.")
else:
    pdp = fetcher.fetch_pdp(PRODUCT_URL)
    print(f"OK  status={pdp.status}  meta={pdp.meta}\n")
    shape(pdp.payload, max_depth=3)

    (cfg.storage.export_dir / "raw_sample_pdp.json").write_text(
        json.dumps(pdp.payload, indent=2, ensure_ascii=False), encoding="utf-8"
    )

In [ ]:
# Cari field teks panjang di response PDP — kandidat deskripsi produk.
def find_long_text(obj, path="", min_len=100, found=None):
    found = [] if found is None else found
    if isinstance(obj, str) and len(obj) >= min_len:
        found.append((path, len(obj), obj[:120].replace("\n", " ")))
    elif isinstance(obj, dict):
        for key, value in obj.items():
            find_long_text(value, f"{path}.{key}" if path else key, min_len, found)
    elif isinstance(obj, list):
        for i, item in enumerate(obj[:5]):
            find_long_text(item, f"{path}.{i}", min_len, found)
    return found


if PRODUCT_URL:
    for path, length, preview in sorted(
        find_long_text(pdp.payload), key=lambda x: x[1], reverse=True
    )[:10]:
        print(f"{length:>6}  {path}")
        print(f"        {preview}...\n")
    print("^ Path terpanjang biasanya deskripsi produk. Kirim blok ini juga ke Claude.")

In [ ]:
fetcher.close()
print("session ditutup")

## Checklist sebelum lanjut ke pipeline

- [ ] Sel 2 (search) sukses, `status=200`
- [ ] Sel 3 menemukan array produk dengan field judul, harga, URL, id
- [ ] `paging` di capture bukan `mode: none` — kalau `none`, hanya halaman 1 yang bisa diambil
- [ ] Stage `pdp` ada dan menemukan field teks panjang (deskripsi)
- [ ] File `capture_*.txt` sudah dihapus

Kalau semua tercentang: kirim output sel **3** dan **5** ke Claude, lalu lanjut fase E
(`parsers.py`, `pipeline.py`, `main.py`).

### Kalau gagal

| Gejala | Tindakan |
|---|---|
| `CaptureIncomplete` | file capture belum ada/tidak lengkap — `docs/CAPTURE_HEADERS.md` |
| `FetchError status=403` | cookie kadaluwarsa — capture ulang langkah A-E |
| `FetchError status=404` | Tokopedia mengganti nama query — capture ulang langkah A-B |
| status 200 tapi nol kandidat array | salah request yang di-capture — langkah B4 |
| `MissingCredential` | `.env` kosong untuk header yang diminta capture |